# 🏪 Laboratorio integrador: del dato a la decisión con el caso Mayoristas

## 📖 La historia

Un **distribuidor mayorista de alimentos en Portugal** vende a 440 clientes: hoteles, restaurantes y cafés (el canal **Horeca**) y tiendas de autoservicio y abarrotes (el canal **Retail**). Hoy los atiende a todos igual: el mismo vendedor, el mismo catálogo y el mismo camión.

El director comercial sospecha que eso le cuesta dinero: un restaurante que compra pescado fresco todos los días no necesita lo mismo que una tienda que compra cajas de detergente cada mes. Te entrega una tabla con el **gasto anual de cada cliente en seis categorías** y te pide una propuesta.

Este es el **cierre del curso**: vas a usar, en un solo proyecto, todo lo que aprendiste. Transformar y escalar (sesión 0), reducir con PCA (sesión 3), agrupar con K-Means (sesión 1), detectar atípicos con DBSCAN (sesión 2) y, sobre todo, **traducir los clusters a decisiones**.

> ### ❓ ¿Cómo debe organizar el distribuidor a sus clientes mayoristas: qué equipos de venta, qué surtido y qué logística le tocan a cada grupo?

## 🧭 Lo que vas a aprender

1. A reconocer datos de gasto **muy sesgados** y por qué conviene aplicar `log1p` **antes** de escalar.
2. A usar **PCA** como paso intermedio: resumir 6 categorías en 2 ejes que se pueden interpretar y graficar.
3. A combinar **K-Means** (segmentos) y **DBSCAN** (clientes atípicos) sobre el mismo espacio.
4. A **validar** un agrupamiento con una etiqueta que **no** usaste para agrupar (el canal de venta).
5. A convertir cada segmento en una **recomendación de negocio** con números en las unidades originales.

## ⏱️ Agenda (3 horas)

| Parte | Tema | Tiempo |
|---|---|---|
| 0 | Preparación, recap del curso y carga de datos | 10 min |
| 1 | Exploración: distribuciones sesgadas y outliers enormes | 20 min |
| 2 | ¿Por qué `log1p` antes de escalar? | 20 min |
| 3 | PCA: de 6 categorías a 2 ejes con significado | 25 min |
| ☕ | Descanso | 10 min |
| 4 | K-Means sobre los componentes principales | 25 min |
| 5 | DBSCAN: ¿quién no encaja en ningún segmento? | 20 min |
| 6 | Perfil de los segmentos en unidades originales | 20 min |
| 7 | Validación contra el canal de venta | 15 min |
| 8 | Recomendaciones de negocio y cierre | 15 min |
| | **Total** | **180 min** |

## 📋 Las variables

Datos: *Wholesale customers* (UCI Machine Learning Repository, Cardoso, 2014; licencia CC BY 4.0). Los gastos están en **unidades monetarias anuales** (u.m.).

| Variable (original) | En el notebook | Descripción | Unidades |
|---|---|---|---|
| `Channel` | `canal` | Horeca (hoteles, restaurantes, cafés) o Retail | categoría |
| `Region` | `region` | Lisboa, Oporto u Otra | categoría |
| `Fresh` | `frescos` | Gasto anual en productos frescos (frutas, verduras, carne, pescado) | u.m./año |
| `Milk` | `lacteos` | Gasto anual en lácteos | u.m./año |
| `Grocery` | `abarrotes` | Gasto anual en abarrotes (productos secos de anaquel) | u.m./año |
| `Frozen` | `congelados` | Gasto anual en congelados | u.m./año |
| `Detergents_Paper` | `detergentes_papel` | Gasto anual en detergentes y productos de papel | u.m./año |
| `Delicassen` | `delicatessen` | Gasto anual en delicatessen (embutidos, quesos finos) | u.m./año |

> ⚠️ **Regla del caso:** `canal` y `region` **no** entran al agrupamiento. Sólo agrupamos con los seis gastos. El canal lo guardamos para el final, como "examen" de nuestros segmentos.

---
# 0️⃣ Preparación (10 min)

## 📋 Recap: lo que ya sabes de las sesiones 1 a 3

| Sesión | Técnica | La idea clave | La trampa que aprendiste a evitar |
|---|---|---|---|
| 1 | **K-Means** | Asigna cada punto al centroide más cercano y recalcula; tú eliges `k` | Elegir `k` con una sola métrica; olvidar `n_init` y `random_state` |
| 2 | **DBSCAN** | Un cluster es una zona densa; lo que no cabe en ninguna es **ruido** (-1) | Elegir `eps` al azar en vez de usar la gráfica de k-distancia |
| 3 | **PCA** | Rota los datos para que pocos ejes (componentes) concentren la varianza | Aplicarlo sin escalar; no leer las cargas |
| Todas | **Escalado** | Las distancias y la varianza dependen de las unidades | Agrupar con variables en escalas distintas |

Hoy los encadenamos: **preparar → reducir → agrupar → validar → interpretar → actuar**.

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_clusters(df, x, y, labels, centroides=None)` | Puntos en 2D coloreados por cluster (el ruido en gris) |
| `plot_clusters_3d(df, x, y, z, labels)` | Lo mismo en 3D (se puede rotar) |
| `plot_codo(ks, inercias)` | Método del codo para elegir k |
| `plot_metricas_k(tabla)` | Silhouette, Davies-Bouldin y Calinski-Harabasz según k |
| `plot_silueta(X, labels)` | Coeficiente de silueta de cada punto, por cluster |
| `plot_k_distancia(X, k, eps=None)` | Distancia al k-ésimo vecino: ayuda a elegir `eps` de DBSCAN |
| `plot_varianza_explicada(pca)` | Varianza explicada por componente y acumulada |
| `plot_cargas(pca, columnas)` | Peso de cada variable original en cada componente |
| `plot_biplot(scores, pca, columnas, labels=None)` | Puntos en PC1-PC2 con una flecha por variable |

In [ ]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _etiquetas_texto(labels):
    """Convierte etiquetas de cluster a texto para que plotly use colores discretos; -1 = 'Ruido'."""
    labels = np.asarray(labels)
    return np.where(labels == -1, 'Ruido', np.char.add('Cluster ', labels.astype(str)))


def _orden_clusters(texto):
    """Orden de la leyenda: Cluster 0, 1, 2, … y 'Ruido' al final."""
    unicos = sorted(set(texto), key=lambda t: (t == 'Ruido', int(t.split()[-1]) if t != 'Ruido' else 0))
    return {'cluster': unicos}


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas'
    )
    fig.update_layout(width=1000, height=1000, title_font_size=20)
    fig.update_traces(diagonal_visible=True, marker={'size': 4, 'opacity': 0.6})
    fig.show()


def plot_clusters(df, x, y, labels, centroides=None, title='Clusters'):
    """Dispersión 2D coloreada por cluster. El ruido de DBSCAN (-1) aparece en gris.
    centroides: arreglo (k, 2) en las mismas unidades que x, y (opcional)."""
    datos = df[[x, y]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter(
        datos,
        x=x,
        y=y,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    if centroides is not None:
        centroides = np.asarray(centroides)
        fig.add_trace(go.Scatter(
            x=centroides[:, 0],
            y=centroides[:, 1],
            mode='markers',
            marker={'symbol': 'x', 'size': 16, 'color': 'black', 'line': {'width': 2}},
            name='Centroides'
        ))
    fig.update_layout(width=800, height=550)
    fig.show()


def plot_clusters_3d(df, x, y, z, labels, title='Clusters en 3D'):
    """Dispersión 3D coloreada por cluster (se puede rotar con el mouse)."""
    datos = df[[x, y, z]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter_3d(
        datos,
        x=x,
        y=y,
        z=z,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    fig.update_traces(marker={'size': 4})
    fig.update_layout(width=850, height=650)
    fig.show()


def plot_codo(ks, inercias):
    """Método del codo: inercia (suma de distancias² al centroide) para cada k."""
    fig = px.line(
        x=list(ks),
        y=list(inercias),
        markers=True,
        labels={'x': 'Número de clusters (k)', 'y': 'Inercia'},
        title='Método del codo'
    )
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_metricas_k(tabla):
    """Silhouette, Davies-Bouldin y Calinski-Harabasz para cada k (tabla de tabla_metricas_k)."""
    metricas = [
        ('silhouette', 'Silhouette (↑ mejor)'),
        ('davies_bouldin', 'Davies-Bouldin (↓ mejor)'),
        ('calinski_harabasz', 'Calinski-Harabasz (↑ mejor)'),
    ]
    fig = make_subplots(rows=1, cols=3, subplot_titles=[t for _, t in metricas])
    for i, (col, _) in enumerate(metricas, start=1):
        fig.add_trace(go.Scatter(x=tabla['k'], y=tabla[col], mode='lines+markers', name=col), row=1, col=i)
        fig.update_xaxes(title_text='k', dtick=1, row=1, col=i)
    fig.update_layout(width=1150, height=400, showlegend=False, title='Métricas internas según k')
    fig.show()


def plot_silueta(X, labels, title=None):
    """Gráfica de silueta: el ancho de cada barra es el coeficiente de un punto, agrupado por cluster.
    La línea punteada es el promedio. Los puntos de ruido (-1) se excluyen."""
    from sklearn.metrics import silhouette_samples, silhouette_score

    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    X, labels = X[mask], labels[mask]
    valores = silhouette_samples(X, labels)
    promedio = silhouette_score(X, labels)

    fig = go.Figure()
    y_inf = 10
    for c in np.unique(labels):
        v = np.sort(valores[labels == c])
        y = np.arange(y_inf, y_inf + len(v))
        fig.add_trace(go.Scatter(
            x=v, y=y, fill='tozerox', mode='lines', line={'width': 0.5}, name=f'Cluster {c}'
        ))
        y_inf += len(v) + 10
    fig.add_vline(x=promedio, line_dash='dash', line_color='red',
                  annotation_text=f'promedio = {promedio:.3f}', annotation_position='top')
    fig.update_layout(
        title=title or f'Gráfica de silueta (k = {len(np.unique(labels))})',
        xaxis_title='Coeficiente de silueta',
        yaxis={'showticklabels': False, 'title': 'Puntos (ordenados por cluster)'},
        width=750,
        height=500
    )
    fig.show()


def plot_k_distancia(X, k, eps=None):
    """Distancia que necesita cada punto para tener k vecinos (contándose a sí mismo), ordenada.
    Usa k = min_samples: igual que DBSCAN, el punto cuenta como uno de sus k vecinos, así que un punto
    es núcleo con un eps dado si su distancia en esta curva es <= eps. El 'codo' sugiere un buen eps.
    eps (opcional) dibuja una línea horizontal de referencia."""
    from sklearn.neighbors import NearestNeighbors

    # n_neighbors=k incluye al propio punto (distancia 0), como min_samples en DBSCAN
    vecinos = NearestNeighbors(n_neighbors=k).fit(X)
    distancias, _ = vecinos.kneighbors(X)
    d_k = np.sort(distancias[:, -1])

    fig = px.line(
        x=np.arange(len(d_k)),
        y=d_k,
        labels={'x': 'Puntos ordenados por distancia', 'y': f'Distancia para tener {k} vecinos (incluido él)'},
        title=f'Gráfica de k-distancia (k = {k})'
    )
    if eps is not None:
        fig.add_hline(y=eps, line_dash='dash', line_color='red',
                      annotation_text=f'eps = {eps}', annotation_position='top left')
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_varianza_explicada(pca):
    """Scree plot: varianza explicada por componente (barras) y acumulada (línea)."""
    ratio = pca.explained_variance_ratio_
    n = np.arange(1, len(ratio) + 1)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=n, y=ratio, name='Por componente'))
    fig.add_trace(go.Scatter(x=n, y=np.cumsum(ratio), mode='lines+markers', name='Acumulada'))
    for umbral in (0.80, 0.95):
        fig.add_hline(y=umbral, line_dash='dot', line_color='gray',
                      annotation_text=f'{umbral:.0%}', annotation_position='right')
    fig.update_layout(
        title='Varianza explicada por componente',
        xaxis={'title': 'Componente principal', 'dtick': 1 if len(ratio) <= 30 else None},
        yaxis={'title': 'Proporción de varianza', 'tickformat': '.0%', 'range': [0, 1.05]},
        width=800,
        height=450
    )
    fig.show()


def plot_cargas(pca, columnas, n_componentes=None):
    """Mapa de calor de las cargas (loadings): cuánto pesa cada variable original en cada componente."""
    n = n_componentes or pca.n_components_
    cargas = pd.DataFrame(
        pca.components_[:n].T,
        index=columnas,
        columns=[f'PC{i + 1}' for i in range(n)]
    ).round(2)
    fig = px.imshow(
        cargas,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        aspect='auto',
        title='Cargas de cada variable en los componentes'
    )
    fig.update_layout(width=200 + 90 * n, height=150 + 40 * len(columnas))
    fig.show()


def plot_biplot(scores, pca, columnas, labels=None, escala=None, title='Biplot PC1 vs PC2'):
    """Puntos en el plano PC1-PC2 más una flecha por variable original (sus cargas).
    scores: salida de pca.transform(X). labels (opcional) colorea los puntos por cluster o grupo."""
    scores = np.asarray(scores)
    datos = pd.DataFrame({'PC1': scores[:, 0], 'PC2': scores[:, 1]})
    if labels is not None:
        datos['grupo'] = _etiquetas_texto(labels) if np.issubdtype(np.asarray(labels).dtype, np.integer) \
            else np.asarray(labels).astype(str)
    fig = px.scatter(datos, x='PC1', y='PC2', color='grupo' if labels is not None else None,
                     opacity=0.6, title=title)
    cargas = pca.components_[:2].T
    escala = escala or 0.9 * np.abs(scores[:, :2]).max() / np.abs(cargas).max()
    for (cx, cy), nombre in zip(cargas * escala, columnas):
        fig.add_annotation(x=cx, y=cy, ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
                           showarrow=True, arrowhead=2, arrowwidth=1.5, arrowcolor='black')
        fig.add_annotation(x=cx * 1.08, y=cy * 1.08, text=nombre, showarrow=False, font={'size': 12})
    r = pca.explained_variance_ratio_
    fig.update_layout(
        xaxis_title=f'PC1 ({r[0]:.1%} de la varianza)',
        yaxis_title=f'PC2 ({r[1]:.1%} de la varianza)',
        width=850,
        height=650
    )
    fig.show()

### 🧰 Funciones de clustering

Hacen en una línea lo que de otra forma repetirías con cada modelo:

| Función | ¿Para qué sirve? |
|---|---|
| `evaluar_clustering(nombre, X, labels)` | Número de clusters, % de ruido, silhouette, Davies-Bouldin y Calinski-Harabasz (sin contar el ruido) |
| `tabla_metricas_k(X, ks)` | Ajusta K-Means para cada k y regresa inercia y métricas en una tabla |
| `perfil_clusters(df, labels, columnas)` | Tamaño y promedio de cada variable por cluster |

In [ ]:
# Funciones de clustering
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score


def evaluar_clustering(nombre, X, labels):
    """Métricas internas de un agrupamiento. El ruido (-1) se excluye de las métricas,
    pero se reporta qué porcentaje de los puntos quedó como ruido."""
    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = len(np.unique(labels[mask]))
    fila = {
        'modelo': nombre,
        'n_clusters': n_clusters,
        'pct_ruido': float(round(100 * (~mask).mean(), 1)),
    }
    if n_clusters >= 2:
        fila['silhouette'] = float(round(silhouette_score(X[mask], labels[mask]), 3))
        fila['davies_bouldin'] = float(round(davies_bouldin_score(X[mask], labels[mask]), 3))
        fila['calinski_harabasz'] = float(round(calinski_harabasz_score(X[mask], labels[mask]), 1))
    else:
        fila.update({'silhouette': np.nan, 'davies_bouldin': np.nan, 'calinski_harabasz': np.nan})
    return fila


def tabla_metricas_k(X, ks, random_state=42):
    """Ajusta K-Means para cada k y regresa una tabla con inercia, silhouette, Davies-Bouldin y Calinski-Harabasz."""
    filas = []
    for k in ks:
        modelo = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(X)
        fila = evaluar_clustering(f'k={k}', X, modelo.labels_)
        filas.append({'k': k, 'inercia': float(round(modelo.inertia_, 1)), **{
            m: fila[m] for m in ('silhouette', 'davies_bouldin', 'calinski_harabasz')}})
    return pd.DataFrame(filas)


def perfil_clusters(df, labels, columnas):
    """Promedio de cada variable por cluster, con tamaño y porcentaje. Sirve para ponerle nombre a cada grupo."""
    datos = df[columnas].copy()
    datos['cluster'] = np.asarray(labels)
    perfil = datos.groupby('cluster')[columnas].mean().round(1)
    perfil.insert(0, 'n', datos['cluster'].value_counts().sort_index())
    perfil.insert(1, 'pct', (100 * perfil['n'] / perfil['n'].sum()).round(1))
    perfil.index = ['Ruido' if c == -1 else f'Cluster {c}' for c in perfil.index]
    return perfil

### 📥 Cargar los datos

El archivo viene en la carpeta `datos/` junto a este notebook. Renombramos las columnas al español y traducimos los códigos de `canal` y `region` para leerlos más fácil.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
import pandas as pd
import numpy as np

df = pd.read_csv('datos/wholesale_customers.csv')

df = df.rename(columns={
    'Channel': 'canal',
    'Region': 'region',
    'Fresh': 'frescos',
    'Milk': 'lacteos',
    'Grocery': 'abarrotes',
    'Frozen': 'congelados',
    'Detergents_Paper': 'detergentes_papel',
    'Delicassen': 'delicatessen',
})
df['canal'] = df['canal'].map({1: 'Horeca', 2: 'Retail'})
df['region'] = df['region'].map({1: 'Lisboa', 2: 'Oporto', 3: 'Otra'})

# Las seis variables de gasto: son las ÚNICAS que entran al agrupamiento
gasto = ['frescos', 'lacteos', 'abarrotes', 'congelados', 'detergentes_papel', 'delicatessen']

df.head()

---
# 1️⃣ Exploración: distribuciones sesgadas y outliers enormes (20 min)

Antes de agrupar hay que conocer los datos: cuántos son, si hay nulos, en qué rango se mueve cada gasto y cómo se relacionan entre sí.

In [ ]:
# ¿Cuántas filas y columnas? ¿Hay valores nulos?
# Pista: df.shape y df.isnull().sum()
# ✍️ Tu código aquí


In [ ]:
# Estadísticas descriptivas de los gastos
# Pista: df[gasto].describe().round(0)
# ✍️ Tu código aquí


In [ ]:
# ¿Cuántos clientes hay por canal y por región?
# Pista: pd.crosstab(df['region'], df['canal'], margins=True)
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Hay nulos? Compara la media y la mediana (50%) de cada gasto: ¿qué te dice que la media sea mucho mayor? ¿Cuánto vale el máximo comparado con la mediana? ¿Qué canal domina?

RELLENAR CON TUS COMENTARIOS

In [ ]:
# Histograma + boxplot de cada gasto
# Pista: plot_distributions(df, gasto)
# ✍️ Tu código aquí


In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
# Dos números que resumen el sesgo de cada variable
sesgo = pd.DataFrame({
    'asimetria': df[gasto].skew().round(2),          # 0 = simétrica; > 1 = muy sesgada a la derecha
    'max_entre_mediana': (df[gasto].max() / df[gasto].median()).round(1),
})
sesgo

✍️ **¿Qué observas?** ¿Qué forma tienen los histogramas? ¿Qué variable es la más sesgada? ¿Cuántas veces la mediana gasta el cliente más grande? Si calculáramos distancias con estos datos, ¿qué clientes dominarían?

RELLENAR CON TUS COMENTARIOS

In [ ]:
# Matriz de correlación entre los gastos
# Pista: plot_correlation_matrix(df, gasto)
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Qué pares de variables están más correlacionados? ¿Ves un "bloque" de variables que suben juntas? ¿Qué tipo de negocio compraría esa canasta? ¿Por qué esto anticipa que PCA servirá?

RELLENAR CON TUS COMENTARIOS

---
# 2️⃣ ¿Por qué `log1p` antes de escalar? (20 min)

`StandardScaler` resta la media y divide entre la desviación estándar, pero **no cambia la forma** de la distribución: un cliente que estaba 16 desviaciones arriba sigue igual de lejos. Para datos de dinero, el truco clásico es aplicar el **logaritmo** antes de escalar:

$$x' = \log(1 + x)$$

- El logaritmo **comprime la cola**: pasar de 1,000 a 10,000 pesa lo mismo que pasar de 10,000 a 100,000 (multiplicar por 10).
- Usamos `log1p` (es decir, $\log(1+x)$) en vez de `log` para que un gasto de 0 no dé $-\infty$.
- En lenguaje de negocio: comparamos **proporciones** ("gasta el doble") en vez de diferencias absolutas ("gasta 5,000 más").

Vamos a escalar **de las dos formas** y comparar.

In [ ]:
# 1. Crea df_log aplicando np.log1p a df[gasto] (el df original NO se modifica)
# 2. Grafica sus distribuciones con plot_distributions(df_log, gasto)
# ✍️ Tu código aquí


In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
# Asimetría antes y después del logaritmo
pd.DataFrame({
    'asimetria_original': df[gasto].skew().round(2),
    'asimetria_log': df_log.skew().round(2),
})

✍️ **¿Qué observas?** ¿Cómo cambió la forma de los histogramas? Compara la asimetría antes y después. ¿Alguna variable quedó sesgada hacia el otro lado? ¿Qué clientes crees que están ahora en esa cola izquierda?

RELLENAR CON TUS COMENTARIOS

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. scaler_crudo = StandardScaler();  X_crudo = scaler_crudo.fit_transform(df[gasto])
# 2. scaler_log = StandardScaler();    X_log = scaler_log.fit_transform(df_log)
# 3. Imprime X_crudo.shape y X_log.shape
# ✍️ Tu código aquí


In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
# ¿Qué tan lejos queda el cliente más extremo de cada variable? (en desviaciones estándar)
pd.DataFrame({
    'max_|z|_crudo': np.abs(X_crudo).max(axis=0).round(1),
    'max_|z|_log': np.abs(X_log).max(axis=0).round(1),
    'celdas_|z|>3_crudo': (np.abs(X_crudo) > 3).sum(axis=0),
    'celdas_|z|>3_log': (np.abs(X_log) > 3).sum(axis=0),
}, index=gasto)

### 🧪 Prueba rápida: ¿cambia el agrupamiento?

Ajustamos un K-Means con k = 4 en cada espacio (un valor cualquiera, sólo para comparar; el k definitivo lo elegimos en la Parte 4) y vemos **cuántos clientes caen en cada cluster**.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
from sklearn.cluster import KMeans

kmeans_crudo = KMeans(n_clusters=4, n_init=10, random_state=42)
labels_crudo = kmeans_crudo.fit_predict(X_crudo)

print('Tamaños con datos crudos:', np.bincount(labels_crudo))
print('Silhouette (espacio crudo):', evaluar_clustering('crudo', X_crudo, labels_crudo)['silhouette'])

In [ ]:
# Haz lo mismo con X_log:
# 1. kmeans_log = KMeans(n_clusters=4, n_init=10, random_state=42)
# 2. labels_log = kmeans_log.fit_predict(X_log)
# 3. Imprime los tamaños con np.bincount(labels_log)
# 4. Imprime la silhouette: evaluar_clustering('log', X_log, labels_log)['silhouette']
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Qué tan lejos queda el cliente más extremo en cada versión? Compara los tamaños de los clusters: ¿cuál te serviría para organizar equipos de venta? La silhouette del espacio crudo sale más alta: ¿qué está premiando? ¿Por qué no deberías compararla con la silhouette calculada en el espacio log?

RELLENAR CON TUS COMENTARIOS

---
# 3️⃣ PCA: de 6 categorías a 2 ejes con significado (25 min)

Vimos que varias categorías se mueven juntas. PCA busca las direcciones que concentran la mayor varianza; si dos o tres ejes resumen casi todo, podemos **agrupar y graficar** en ese espacio reducido. Primero ajustamos PCA con **todos** los componentes para decidir cuántos quedarnos.

In [ ]:
from sklearn.decomposition import PCA

# 1. pca_log = PCA() y ajústalo con pca_log.fit(X_log)
# 2. Imprime pca_log.explained_variance_ratio_ y su acumulada con np.cumsum(...)
# 3. Grafica con plot_varianza_explicada(pca_log)
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Cuánta varianza explican PC1 y PC2 juntos? ¿Dónde ves el codo? ¿Cuántos componentes necesitarías para el 80%? ¿Con cuántos te quedarías y por qué?

RELLENAR CON TUS COMENTARIOS

In [ ]:
# ¿Qué variables pesan en cada componente?
# Pista: plot_cargas(pca_log, gasto, n_componentes=3)
# ✍️ Tu código aquí


✍️ **¿Qué observas?** Completa la tabla con las variables que más pesan en cada componente y ponle un nombre de negocio a cada eje.

| Componente | Variables con más peso | Nombre de negocio |
|---|---|---|
| PC1 | ______ | ______ |
| PC2 | ______ | ______ |

RELLENAR CON TUS COMENTARIOS

In [ ]:
# 1. pca_final = PCA(n_components=<los componentes que elegiste>)
# 2. X_pcs = pca_final.fit_transform(X_log)
# 3. df_pcs = pd.DataFrame(X_pcs, columns=[f'PC{i + 1}' for i in range(X_pcs.shape[1])], index=df.index)
# 4. Grafica con plot_biplot(X_pcs, pca_final, gasto)   (dibuja PC1 contra PC2)
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Hacia dónde apuntan las flechas de cada canasta? ¿Ves grupos claramente separados en la nube o una sola masa? ¿Hay clientes lejos del resto? ¿Qué significa estar arriba a la derecha del biplot?

RELLENAR CON TUS COMENTARIOS

---
# ☕ Descanso (10 min)

---
# 4️⃣ K-Means sobre los componentes principales (25 min)

Ahora sí, agrupamos en el espacio `X_pcs` (los componentes que elegiste). Como en la sesión 1, elegimos `k` con **varias señales**: codo, silhouette, Davies-Bouldin y Calinski-Harabasz... y con **sentido de negocio**.

In [ ]:
# Métricas de K-Means para k = 2 ... 8 en el espacio de componentes
# Pista: tabla_k = tabla_metricas_k(X_pcs, range(2, 9))
# ✍️ Tu código aquí


In [ ]:
# 1. Grafica el codo: plot_codo(tabla_k['k'], tabla_k['inercia'])
# 2. Grafica las métricas: plot_metricas_k(tabla_k)
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Dónde ves el codo? ¿Qué k prefiere cada métrica? ¿Coinciden? Si las métricas casi empatan, ¿qué criterio usarías para decidir? Escribe el k que eliges y por qué.

RELLENAR CON TUS COMENTARIOS

In [ ]:
# 1. kmeans_final = KMeans(n_clusters=<tu k>, n_init=10, random_state=42)
# 2. labels_final = kmeans_final.fit_predict(X_pcs)
# 3. Imprime los tamaños con np.bincount(labels_final)
# 4. plot_clusters(df_pcs, 'PC1', 'PC2', labels_final, centroides=kmeans_final.cluster_centers_)
# ✍️ Tu código aquí


In [ ]:
# Gráfica de silueta de tu modelo
# Pista: plot_silueta(X_pcs, labels_final)
# ✍️ Tu código aquí


In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
from sklearn.metrics import adjusted_rand_score

# ¿Perdimos algo por agrupar en los componentes principales y no en las 6 variables (log + escaladas)?
kmeans_6d = KMeans(n_clusters=kmeans_final.n_clusters, n_init=10, random_state=42)
labels_6d = kmeans_6d.fit_predict(X_log)

print('Coincidencia (ARI) entre agrupar en los PCs y en las 6 variables:',
      round(adjusted_rand_score(labels_final, labels_6d), 3))

✍️ **¿Qué observas?** ¿Cuántos clientes quedaron en cada cluster? ¿Dónde está cada cluster en el plano PC1-PC2 y qué significa eso en términos de canastas? ¿La silueta es alta o moderada? ¿Por qué? ¿Perdimos información al agrupar con los componentes en vez de las 6 variables?

RELLENAR CON TUS COMENTARIOS

### 📊 Guardamos los resultados

In [ ]:
# 1. comparacion = []
# 2. comparacion.append(evaluar_clustering('K-Means final (PCs)', X_pcs, labels_final))
# ✍️ Tu código aquí


---
# 5️⃣ DBSCAN: ¿quién no encaja en ningún segmento? (20 min)

K-Means **obliga** a cada cliente a pertenecer a un cluster, aunque esté lejísimos de todos. DBSCAN no: lo que no está en una zona densa queda como **ruido**. Lo usamos sobre el **mismo espacio** `X_pcs` para encontrar clientes atípicos.

Receta de la sesión 2:
1. `min_samples` ≈ 2 × número de dimensiones. Calcúlalo con los componentes que elegiste: `min_samples = 2 * X_pcs.shape[1]`.
2. `eps` sale de la **gráfica de k-distancia** con k = `min_samples`: buscamos el codo donde la curva se dispara.

In [ ]:
# 1. min_samples = 2 * X_pcs.shape[1]
# 2. Grafica la k-distancia: plot_k_distancia(X_pcs, k=min_samples)
# 3. Ubica el codo (donde la curva se dispara) y vuelve a graficar con eps=<tu eps> para verificar
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿En qué valor de la distancia empieza a dispararse la curva? ¿Qué `eps` elegiste? ¿Qué porcentaje aproximado de clientes queda por encima de tu línea?

RELLENAR CON TUS COMENTARIOS

In [ ]:
from sklearn.cluster import DBSCAN

# 1. dbscan_atipicos = DBSCAN(eps=<tu eps>, min_samples=min_samples)
# 2. labels_db = dbscan_atipicos.fit_predict(X_pcs)
# 3. fila_db = evaluar_clustering('DBSCAN (PCs)', X_pcs, labels_db)
# 4. comparacion.append(fila_db) y muestra fila_db
# ✍️ Tu código aquí


In [ ]:
# Grafica el resultado de DBSCAN (el ruido sale en gris)
# Pista: plot_clusters(df_pcs, 'PC1', 'PC2', labels_db, title='DBSCAN sobre PC1 y PC2')
# ✍️ Tu código aquí


In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
# Sensibilidad: ¿qué pasa con otros valores de eps?
pd.DataFrame([
    evaluar_clustering(f'eps={eps}', X_pcs, DBSCAN(eps=eps, min_samples=dbscan_atipicos.min_samples).fit_predict(X_pcs))
    for eps in [0.4, 0.6, 0.8, 1.0, 1.2]
])

✍️ **¿Qué observas?** ¿Cuántos clusters encontró DBSCAN y qué porcentaje es ruido? ¿Por qué algunas métricas salen `NaN`? ¿Qué te dice esto sobre si los clientes forman grupos separados o un continuo? Según la tabla de sensibilidad, ¿qué tan estable es el porcentaje de ruido?

RELLENAR CON TUS COMENTARIOS

### 🔎 ¿Quiénes son los atípicos?

Miramos a los clientes marcados como ruido **en unidades originales** (u.m. al año), ordenados por su gasto total.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
df['gasto_total'] = df[gasto].sum(axis=1)

atipicos = df[labels_db == -1].sort_values('gasto_total', ascending=False)
atipicos[gasto + ['gasto_total', 'canal']]

✍️ **¿Qué observas?** Mira a los clientes atípicos: ¿todos son grandes compradores? ¿Ves algún patrón distinto entre ellos? ¿Qué tipos de atípicos distingues? ¿Por qué el logaritmo hizo visibles a algunos de ellos?

RELLENAR CON TUS COMENTARIOS

---
# 6️⃣ Perfil de los segmentos en unidades originales (20 min)

Un cluster sólo es útil si se puede **explicar en el idioma del negocio**. Nadie en el área comercial entiende "PC1 = 1.18": hay que regresar a **u.m. al año**. Por eso aplicamos `perfil_clusters` sobre el `df` **original**, sin logaritmo ni escalado.

In [ ]:
# Perfil de cada cluster en unidades originales (u.m. al año)
# Pista: perfil_final = perfil_clusters(df, labels_final, gasto + ['gasto_total'])
# ✍️ Tu código aquí


In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
# Como los gastos son muy sesgados, la mediana describe mejor al cliente típico de cada cluster
medianas = df[gasto + ['gasto_total']].groupby(labels_final).median().round(0)
medianas.index = [f'Cluster {c}' for c in medianas.index]

# ¿Qué porcentaje de la venta total aporta cada cluster?
medianas['pct_de_la_venta'] = (
    100 * df.groupby(labels_final)['gasto_total'].sum() / df['gasto_total'].sum()
).round(1).values
medianas

✍️ **¿Qué observas?** Para cada cluster: ¿en qué categorías gasta más y en cuáles casi nada? ¿Cuál es el más grande en número de clientes? ¿Cuál aporta más venta? Ponle un **nombre de negocio** a cada segmento.

| Cluster | Nombre del segmento | ¿Qué lo distingue? |
|---|---|---|
| ______ | ______ | ______ |
| ______ | ______ | ______ |

(Agrega una fila por cada cluster de tu modelo.)

RELLENAR CON TUS COMENTARIOS

In [ ]:
# Perfil del ruido de DBSCAN contra el cluster denso
# Pista: perfil_clusters(df, labels_db, gasto + ['gasto_total'])
# ✍️ Tu código aquí


In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
# ¿En qué segmento de K-Means había quedado cada atípico?
pd.crosstab(
    pd.Series(labels_db, name='DBSCAN').map(lambda c: 'Ruido' if c == -1 else 'Denso'),
    pd.Series(labels_final, name='K-Means')
)

✍️ **¿Qué observas?** ¿El promedio del ruido describe bien a los atípicos? ¿Por qué no? ¿En qué clusters de K-Means habían quedado? ¿Qué pasaría si no los separáramos?

RELLENAR CON TUS COMENTARIOS

---
# 7️⃣ Validación contra el canal de venta (15 min)

Ahora sí sacamos la variable que guardamos: `canal` (Horeca o Retail). **No la usamos para agrupar**, así que es una prueba honesta: si nuestros segmentos tienen sentido, deberían relacionarse con el tipo de negocio del cliente.

⚠️ En la mayoría de los proyectos reales **no tienes** una etiqueta así. Aquí tenemos la suerte de poder revisar; aun así, no buscamos que los clusters **copien** el canal: buscamos que lo **expliquen** y, si se puede, que digan algo más.

In [ ]:
# Tabla cruzada de tus clusters contra el canal
# Pista: pd.crosstab(labels_final, df['canal'], margins=True)
# ✍️ Tu código aquí


In [ ]:
# Channel tiene 2 categorías: ajusta también un K-Means con k = 2 para compararlo de tú a tú
# 1. kmeans_k2 = KMeans(n_clusters=2, n_init=10, random_state=42)
# 2. labels_k2 = kmeans_k2.fit_predict(X_pcs)
# 3. comparacion.append(evaluar_clustering('K-Means k=2 (PCs)', X_pcs, labels_k2))
# 4. Tabla cruzada: pd.crosstab(labels_k2, df['canal'], margins=True)
# ✍️ Tu código aquí


In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
# ARI: 1 = los grupos coinciden perfecto con la etiqueta; 0 = no más parecido que al azar
pd.DataFrame({
    'modelo': ['K-Means final (PCs)', 'K-Means k=2 (PCs)', 'DBSCAN (PCs)', 'K-Means final vs REGIÓN'],
    'ARI': [
        adjusted_rand_score(df['canal'], labels_final),
        adjusted_rand_score(df['canal'], labels_k2),
        adjusted_rand_score(df['canal'], labels_db),
        adjusted_rand_score(df['region'], labels_final),
    ],
}).round(3)

In [ ]:
# Dibuja el biplot coloreado por el canal real
# Pista: plot_biplot(X_pcs, pca_final, gasto, labels=df['canal'], title='Biplot coloreado por canal')
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Algún cluster es casi puro de un canal? ¿Cuál de los modelos se parece más al canal según el ARI? Si tu modelo final tiene un ARI menor, ¿significa que está mal? ¿Qué te dice el ARI contra la región sobre organizar a los vendedores por zona? En el biplot, ¿hacia dónde se cargan los Retail y los Horeca?

RELLENAR CON TUS COMENTARIOS

---
# 8️⃣ Recomendaciones de negocio y cierre (15 min)

El último paso del flujo es **actuar**. Para cada segmento decide: qué **equipo de venta** lo atiende, qué **surtido** se le ofrece y qué **logística** necesita. Y no olvides a los atípicos.

### 💼 Recomendaciones por segmento

✍️ Completa la tabla con **tus** segmentos (los nombres que elegiste en la Parte 6) y tus números.

| Segmento | Clientes | % de la venta | Equipo de venta | Surtido | Logística |
|---|---|---|---|---|---|
| ______ | ______ | ______ | ______ | ______ | ______ |
| ______ | ______ | ______ | ______ | ______ | ______ |
| Atípicos (DBSCAN) | ______ | — | ______ | ______ | ______ |

(Agrega una fila por cada segmento de tu modelo.)

✍️ ¿Qué harías con los clientes atípicos? ¿Les darías el mismo trato a todos?

RELLENAR CON TUS COMENTARIOS

### 📊 Tabla comparativa final

In [ ]:
# 1. tabla_comparacion = pd.DataFrame(comparacion)
# 2. Agrega la columna 'ari_canal' con el ARI de cada modelo contra df['canal'],
#    en el mismo orden en que agregaste los modelos a comparacion:
#    [round(adjusted_rand_score(df['canal'], l), 3) for l in (labels_final, labels_db, labels_k2)]
# 3. Muestra tabla_comparacion
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Cuál modelo gana en cada métrica? ¿Por qué sí podemos comparar estas silhouettes entre sí (a diferencia de la del espacio crudo)? ¿Qué aporta DBSCAN si no gana ninguna métrica?

RELLENAR CON TUS COMENTARIOS

## ❓ Respuesta a la pregunta

> **¿Cómo debe organizar el distribuidor a sus clientes mayoristas?**

✍️ Escribe tu respuesta en 4-6 líneas: cuántos segmentos, cómo se llaman, cuántos clientes y qué % de la venta trae cada uno, qué equipo/surtido/logística le toca, qué hacer con los atípicos y si la región o el canal bastarían para organizarlos.

RELLENAR CON TUS COMENTARIOS

## 📝 Lo que aprendimos

✍️ Escribe las 3 a 5 lecciones más importantes de este caso (y del curso).

RELLENAR CON TUS COMENTARIOS

## 🚀 Retos opcionales

1. Repite el análisis con **un componente más** de los que elegiste. ¿Cambian los segmentos?
2. Prueba **K-Means con otro k**. ¿El segmento nuevo es accionable o sólo parte uno existente?
3. Usa **HDBSCAN** (`sklearn.cluster.HDBSCAN`) sobre `X_pcs`. ¿Encuentra los mismos atípicos?
4. Agrupa con **proporciones** en vez de montos: divide cada gasto entre el `gasto_total` del cliente. ¿Aparecen segmentos por *tipo* de canasta sin importar el tamaño?